# Bases de données et création de procédés

Nous avons vu qu'une ACV consistait en deux grandes étapes : 
1. celle d'inventaire ;
2. puis celle de caractérisation d'inventaire.

En pratique, on s'appuie sur des bases de données (BDD) qui décrivent :
- la technosphère (ensemble des procédés et des flux qui les relient, dits flux intermédiaires) ; 
- la biosphère (ensemble des matières et substances potentiellement extraites ou émises par les procédés, dits flux élémentaires) ;
- les méthodes d'impacts (définition des catégories d'impact, des indicateurs, des facteurs de caractérisation associés, et accessoirement des facteurs d'agrégation et/ou de normalisation)

![schéma inventaire-impacts](Images/Hypotheses_ACV_3.png)

Commençons par les procédés. Nous utilisons couramment pour modéliser ces derniers la base de données ecoinvent qui est générique - elle recouvre plusieurs secteurs d'activités (Industrie, Énergie, Matériaux, Transports...).

Vous pouvez explorer cette BDD en ligne à ce lien : [https://ecoquery.ecoinvent.org/3.11/cutoff](https://ecoquery.ecoinvent.org/3.11/cutoff). (version, allocation, exemple de procédé)

Pour réaliser des ACV à partir de ces BDD, plusieurs logiciels sont disponibles (des libres tels OpenLCA et brightway ; des payants tels SimaPro, OneClick LCA ou Holis). Nous utilisons la librairie python brightway 2.5 dans ce qui suit car elle est open-source, permet de profiter de l'écosystème python, s'appuie explicitement sur le formalisme mathématique vu précédemment et évite l'effet "boîte noire".

**Documentation de brightway2.5**

Voir :

- documentation générale : https://docs.brightway.dev/en/latest/content/overview/index.html
- pense-bête : https://docs.brightway.dev/en/latest/content/cheatsheet/index.html

## Import des librairies et mise en place du projet

In [ ]:
import bw2io as bi # ensemble des fonctions et classes pour importer et exporter (input/output)
import bw2data as bd # ... pour gérer les données du projet
import bw2calc as bc # ... pour faire des opérations
import bw2analyzer as ba # ... pour interpréter les résultats

import pandas as pd # pour utiliser un format de table pratique
import seaborn as sns # pour tracer des graphes à partir de ces tables
import matplotlib.pyplot as plt

from pathlib import Path

In [ ]:
# Nom de projet et bases de données requises
PROJECT = "project_ecoinvent_311"
REQUIRED_DATABASES = {"ecoinvent-3.11-biosphere", "ecoinvent-3.11-cutoff"}

# Chemin vers archive contenant ecoinvent 3.11 cutoff
BACKUP_LOCATIONS = [
    # JupyterHub
    Path("/srv/cours-acv-2026/brightway2_project_ecoinvent_311.tar.gz"),

    # Local computer
    Path.cwd().joinpath("brightway2_project_ecoinvent_311.tar.gz")
]
BACKUP = next((path for path in BACKUP_LOCATIONS if path.exists()), None)

# Vérification de l'état du projet
project_is_ready = False
if PROJECT in bd.projects:
    bd.projects.set_current(PROJECT)
    missing_databases = REQUIRED_DATABASES - set(bd.databases)
    if not missing_databases:
        project_is_ready = True
        print(f"Le projet Brightway '{PROJECT}' est prêt.")
    else:
        print(f"Le projet '{PROJECT}' existe mais est incomplet.")
        print("Base(s) de donnée(s) manquante(s):")
        for db in sorted(missing_databases):
            print(f"  - {db}")

# Restore projet si besoin
if not project_is_ready:
    if PROJECT in bd.projects:
        print(f"Suppression du projet incomplet '{PROJECT}'...")

        bd.projects.delete_project(
            PROJECT,
            delete_dir=True
        )
        bd.projects.purge_deleted_directories()

    print(f"Installation du projet depuis:\n{BACKUP}")

    bi.backup.restore_project_directory(
        BACKUP,
        overwrite_existing=True
    )

    missing_databases = REQUIRED_DATABASES - set(bd.databases)

    if missing_databases:
        raise RuntimeError(
            "Projet restoré, mais des bases de données "
            f"manquent toujours: {sorted(missing_databases)}"
        )

    print(f"Le projet '{PROJECT}' a été installé avec succès.")


# Activation du projet
bd.projects.set_current(PROJECT)
print(f"\nProjet activé: {bd.projects.current}")

In [ ]:
bd.databases # On peut vérifier les BDD disponibles dans ce projet

## Exploration d'ecoinvent

On peut commencer par se poser les questions suivantes.

1. Combien y a-t-il d'éléments dans cette biosphère ?

In [ ]:
biodb = bd.Database('ecoinvent-3.11-biosphere')
len(biodb)

In [ ]:
element = biodb.random() #On met dans une variable un élément de la biosphère choisi aléatoirement 
element

In [ ]:
element.as_dict() # brightway permet de regarder sous forme de dictionnaire python toutes les métadonnées de l'élément

#### Exercice

2. Combien y a-t-il de procédés dans cette technosphère ?

Pour créer une cellule de code vous pouvez cliquer sur le symbole "+" en haut du notebook, ou bien cliquer dans la marge du document et appuyer sur la touche "b" (pour "below") ou "a" (pour "above")

##### Correction

In [ ]:
eidb = bd.Database('ecoinvent-3.11-cutoff') #On sélectionne la technosphere que l'on stocke dans une variable
len(eidb) #La longueur de celle-ci correspond au nombre de procédés différents disponibles

#### Suite du TD

In [ ]:
activity = eidb.random()
activity

In [ ]:
activity.as_dict()

3. Comment un procédé est-il relié aux autres ?

In [ ]:
activity = eidb.search('anchovy') #On utilise une fonction de recherche par mot-clé, ici "anchovy"
activity # brightway retourne une liste de procédés

In [ ]:
activity = eidb.search('anchovy, capture by steel purse seiner')[0] #On choisit le premier élément de la liste, en précisant le mot-clé
activity #Ici, activity sera bien un procédé, non une liste de procédés

On regarde maintenant quels sont ses échanges avec la technosphère et la biosphère.

In [ ]:
list(activity.exchanges()) #Tous les échanges

In [ ]:
list(activity.technosphere()) #Flux intermédiaires

In [ ]:
list(activity.biosphere()) #Flux élémentaires

In [ ]:
list(activity.production()) #Flux de référence (produit)

N'oubliez pas la page pense-bête de brightway : https://docs.brightway.dev/en/latest/content/cheatsheet/inventory.html

Pour les personnes souhaitant en savoir plus sur la structure en graphe des données dans brightway : https://docs.brightway.dev/en/latest/content/overview/inventory.html 

Bien que nous adorions la tapenade, nous allons commencer par modéliser ce que tout élève des Ponts préfère : le BÉTON ARMÉ.

## Construction de procédés béton

Il en existe dans la base de données :

In [ ]:
concrete = eidb.search('concrete')
concrete

Existe-t-il des procédés modélisant des bétons spécifiquement français ?

Nous utilisons une compréhension de liste conditionnelle pour préciser la recherche :

In [ ]:
concrete = [a for a in eidb if 'concrete' in a['name'].lower() and a['location'] == 'FR'] # lower met la chaine de caractères en minuscules, ce qui évite la sensibilité à la casse.
concrete

Comme il n'y en a pas, nous allons modéliser deux formulations avec les données dont nous disposons, pour les comparer. On en profite pour prendre en compte des armatures (on étudie du béton armé) et de l'énergie (malaxage du béton).

Considérons les formulations suivantes pour 1 m3 de béton : 

- 350kg de ciment Portland (béton A) ou CEM III/B (béton B)
- 175kg d'eau
- 800kg de sable
- 1100kg de gravier

Pour le ferraillage, prenons un ratio de 100 kg d'acier par m3 de béton. Avez-vous une remarque ?

Pour l'énergie de malaxage, prenons l'hypothèse de 14.4 MJ par m3 de béton.

Les procédés mobilisés sont les suivants :

Béton | Flux | Procédé choisi dans ecoinvent | Région | Unité | Qté pour 1m3
:---: | :---: | :---: | :---: | :---: | :---:
Béton A | ciment Portland | market for cement, Portland | Europe without Switzerland | kg | 350
Béton B | CEM III/B | market for cement, CEM III/B | Europe without Switzerland | kg | 350
Béton A & B | eau | market for tap water | Europe without Switzerland | kg | 175
Béton A & B | sable | market for sand | Rest-of-World (RoW) | kg | 800 
Béton A & B | graviers | market for gravel, crushed | Rest-of-World (RoW) | kg | 1100
Béton A & B | ferraillage | market for reinforcing steel | Global | kg | 100
Béton A & B | énergie | diesel, burned in building machine | Global | MJ | 14.4

In [ ]:
from pathlib import Path
if 'betons_armes' in bd.databases : 
    print('procédés déjà importés !')
else : 
    imp = bi.ExcelImporter(Path.cwd().joinpath("betons_armes.xlsx"))
    imp.apply_strategies() #applique une série de routines sur la base de données, notamment une conversion des variables dans les bons formats
    imp.match_database(fields=('name', 'unit', 'location')) #identifie les échanges que les procédés de la base ont avec d'autres procédés de la même base
    imp.match_database('ecoinvent-3.11-cutoff', fields=('name', 'unit', 'location')) #identifie les échanges que les procédés de la base ont avec les procédés d'ecoinvent
    imp.statistics() #affiche les statistiques des "match" réalisés, notamment si certains procédés n'ont pas été trouvés ("unlinked exchanges") 
    imp.write_database() #écrit et sauvegarde la base de données

In [ ]:
fgdb = bd.Database('betons_armes') #fgdb pour foreground database (procédés d'avant-plan vs arrière-plan)

In [ ]:
beton_A = fgdb.search('Béton A')[0]
beton_A

In [ ]:
list(beton_A.edges()) #brightway utilise indifféremment les termes edges/exchanges et node/activity

## Calcul des impacts environnementaux

Pour calculer des impacts environnementaux, il faut savoir quoi calculer, c'est-à-dire choisir une méthode d'impact. 

Remarque sur le mot méthode :
- Traditionnellement, une méthode d'impact désigne un ensemble de catégories d'impacts, chaque catégorie contenant des indicateurs environnementaux et les facteurs de caractérisation permettant de calculer ces derniers à partir des flux élémentaires (en réalité, pour chaque indicateur, à partir d'un sous-ensemble de la biosphère).
- dans brightway, une "method" désigne en fait un indicateur spécifique (calculé selon une certaine méthode d'impact), ce qui peut prêter à confusion.


Pour ce TD, nous choisissons la méthode d'impacts Environmental Footprint v3.1.

In [ ]:
meth = [m for m in bd.methods if 'EF v3.1' in m[1] and 'no LT' not in m[1]] # no LT signifie no long-term, c'est à dire que les substances émises ou extraites après 100 ans ne sont pas comptées.
meth

### Calcul pour un indicateur et une demande

On s'intéresse dans un premier temps à l'impact sur le changement climatique en considérant le pouvoir de réchauffement global à 100 ans.

In [ ]:
gwp100 = [ind for ind in meth if 'GWP100' in str(ind) and 'biogenic' not in str(ind) and 'fossil' not in str(ind) and 'land' not in str(ind)][0] # On sélectionne l'indicateur par compréhension de liste
gwp100

On fait le calcul pour le béton A :

(Un rappel sur les définitions des différentes matrices vues en cours est disponible dans la documentation de brightway https://learn.brightway.dev/en/latest/content/chapters/BW25/BW25_introduction.html)

In [ ]:
lca = bc.LCA(demand={beton_A : 1.0},method=gwp100) #Préparation de matrices réduites A, B, C
lca.lci()# Préparation du calcul de g. Si on appelle le score à cette étape, on obtient g.
lca.lcia() # Préparation du calcul de h.
lca.score # Ici le calcul se fait, et retourne le score d'impact

et maintenant pour le béton B :

In [ ]:
beton_B = fgdb.search('Béton B')[0]
beton_B

In [ ]:
lca = bc.LCA(demand={beton_B : 1.0},method=gwp100)
lca.lci()
lca.lcia()
lca.score

Vous pouvez consulter les constituants des différents types de ciments dans le tableau issus de la norme, joint à ce TD.

#### Exercice

Quel béton armé est le moins impactant en ce qui concerne l'eutrophisation ?

### Calcul pour plusieurs indicateurs et plusieurs demandes

La logique est strictement la même que pour un indicateur et une demande. La fonction qui gère ce calcul réclame simplement que l'on range les arguments d'une manière spécifique, comme réalisé ci-dessous :

In [ ]:
demands = { 'A' : {beton_A.id : 1.0}, 'B' : {beton_B.id : 1.0}} # Les demandes évaluées sont rangées dans un dictionnaire, elles ont un nom, ici nous choisissons A et B.

method_config = {'impact_categories' : meth} # les indicateurs d'impacts sont aussi rangés dans un dictionnaire
data_objs = bd.get_multilca_data_objs(demands, method_config) # brightway fait sa tambouille

lca = bc.MultiLCA(demands=demands, method_config= method_config, data_objs=data_objs)
lca.lci()
lca.lcia()
results = lca.scores

Les résultats sont stockés dans un dictionnaire dont les clés sont les couples (indicateurs, nom de la demande) et les valeurs sont les scores d'impacts : c'est peu lisible.

In [ ]:
results

On range les résultats dans un tableau pour les voir et les tracer plus aisément.

In [ ]:
df_results = pd.DataFrame.from_dict(results,orient='index') # on met les résultats dans un dataframe
res = pd.DataFrame(columns=['indicateur','unité','beton','score']) #
for r in df_results.index :
    res = pd.concat([pd.DataFrame([[str(r[0][2:3]),bd.Method(r[0]).metadata['unit'],r[1],df_results.at[r,0]]], columns=res.columns), res], ignore_index=True)

In [ ]:
res# Visualition de la table de résultat

On peut ajouter une colonne où l'on normalise par le maximum pour chaque indicateur

In [ ]:
res['score_norm'] = [res.at[i,'score']/max(res[res.indicateur == res.at[i,'indicateur']]['score']) for i in res.index]
res

In [ ]:
# fig, ax = plt.subplots(figsize=(10,5))
g=sns.catplot(data=res,x='indicateur',y='score_norm',kind='bar',hue='beton',errorbar=None)
g.fig.set_size_inches(20, 5);
g.ax.set_xticklabels(g.ax.get_xticklabels(), rotation=45, ha='right');

#### Exercice

Sauriez-vous refaire cette comparaison avec une autre méthode d'impact ? Par exemple ReCiPe v1.03.